In [1]:
!nvidia-smi
import sys, platform, subprocess, re
print("Python:", sys.version)

Tue Aug 19 12:05:12 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# !pip install -U jedi
# !pip -q install -U pip wheel setuptools
# !pip install -U vllm

# 📘 Параметры `vllm.LLM(...)` (с дефолтами)

## Основные

**`model`**  
Имя или путь модели (Hugging Face репозиторий или локальная папка).  
Пример: `Qwen/Qwen2.5-7B-Instruct-AWQ`, `TinyLlama/TinyLlama-1.1B-Chat-v1.0`.  
⚠️ Обязательный параметр, значения по умолчанию нет.

---

**`quantization`**  
Тип квантованной модели.  
- Возможные значения: `None`, `"awq"`, `"gptq"`  
- По умолчанию: `None`  
> Работает только с заранее квантованными моделями.

---

**`dtype`**  
Вычислительный тип весов/активаций.  
- Возможные значения: `"auto"`, `"float16"`, `"bfloat16"`, `"float32"`  
- По умолчанию: `"auto"`  
> На T4 это обычно означает `float16`.

---

**`max_model_len`**  
Максимальная длина контекста (токены).  
- Возможные значения: целое число (например, 1024–4096)  
- По умолчанию: берётся из конфигурации модели (обычно `2048`)  
> Уменьшение снижает расход памяти KV-кэша.

---

**`gpu_memory_utilization`**  
Доля доступной видеопамяти, которую можно занять.  
- Возможные значения: число от `0.0` до `1.0`  
- По умолчанию: `0.90`  
> На T4 безопасный диапазон `0.85–0.90`.

---

## Дополнительные

**`tensor_parallel_size`**  
Количество GPU для тензорного параллелизма.  
- Возможные значения: целое число ≥1  
- По умолчанию: `1`  
> В Colab всегда `1`.

---

**`trust_remote_code`**  
Разрешает запуск кастомного кода из Hugging Face репозитория.  
- Возможные значения: `True`, `False`  
- По умолчанию: `False`  
> Иногда нужно для нестандартных моделей.

---

**`tokenizer`**  
Явное указание токенайзера.  
- По умолчанию: совпадает с `model`.

---

**`tokenizer_mode`**  
Режим загрузки токенайзера.  
- Возможные значения: `"auto"`, `"slow"`, `"hf_tokenizers"`  
- По умолчанию: `"auto"`

---

**`download_dir`**  
Каталог для скачивания весов.  
- По умолчанию: системный кэш Hugging Face (`~/.cache/huggingface`)

---

**`revision`**  
Ветка или коммит модели.  
- По умолчанию: `"main"`

---

**`seed`**  
Сид генерации.  
- Возможные значения: целое число  
- По умолчанию: `None` (случайность без фиксированного сида)

---

**`kv_cache_dtype`**  
Тип данных для KV-кэша.  
- Возможные значения: `"auto"`, `"fp16"`, `"bf16"`  
- По умолчанию: `"auto"`  
> На T4 это обычно `fp16`.


In [3]:
# Запуск Qwen 7B Instruct в AWQ (INT4). Если репо приватное — нужен HF токен.
from vllm import LLM, SamplingParams

model = "Qwen/Qwen2.5-7B-Instruct-AWQ"  # квантованная версия (AWQ int4)
llm = LLM(
    model=model,
    quantization="awq",
    dtype="auto",
    max_model_len=1024,                # урезаем контекст, чтобы T4 не упала в OOM
    gpu_memory_utilization=0.90,       # аккуратно с VRAM
)

INFO 08-19 12:05:24 [__init__.py:241] Automatically detected platform cuda.
INFO 08-19 12:05:25 [utils.py:326] non-default args: {'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ', 'max_model_len': 1024, 'disable_log_stats': True, 'quantization': 'awq'}


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


INFO 08-19 12:05:41 [__init__.py:711] Resolved architecture: Qwen2ForCausalLM
INFO 08-19 12:05:41 [__init__.py:1750] Using max model len 1024
WARNING 08-19 12:05:43 [__init__.py:1171] awq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
WARNING 08-19 12:05:43 [arg_utils.py:1770] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
INFO 08-19 12:05:43 [llm_engine.py:222] Initializing a V0 LLM engine (v0.10.1) with config: model='Qwen/Qwen2.5-7B-Instruct-AWQ', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=awq, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, decoding_config=Deco

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 08-19 12:06:08 [default_loader.py:262] Loading weights took 21.73 seconds
INFO 08-19 12:06:09 [model_runner.py:1112] Model loading took 5.2036 GiB and 22.531418 seconds
INFO 08-19 12:06:12 [worker.py:295] Memory profiling takes 2.45 seconds
INFO 08-19 12:06:12 [worker.py:295] the current vLLM instance can use total_gpu_memory (14.74GiB) x gpu_memory_utilization (0.90) = 13.27GiB
INFO 08-19 12:06:12 [worker.py:295] model weights take 5.20GiB; non_torch_memory takes 0.05GiB; PyTorch activation peak memory takes 1.40GiB; the rest of the memory reserved for KV Cache is 6.61GiB.
INFO 08-19 12:06:13 [executor_base.py:114] # cuda blocks: 7741, # CPU blocks: 4681
INFO 08-19 12:06:13 [executor_base.py:119] Maximum concurrency for 1024 tokens per request: 120.95x
INFO 08-19 12:06:17 [model_runner.py:1383] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in

Capturing CUDA graph shapes:   0%|          | 0/35 [00:00<?, ?it/s]

INFO 08-19 12:07:11 [model_runner.py:1535] Graph capturing finished in 54 secs, took 0.50 GiB
INFO 08-19 12:07:11 [llm_engine.py:417] init engine (profile, create kv cache, warmup model) took 61.20 seconds
INFO 08-19 12:07:11 [llm.py:298] Supported_tasks: ['generate']


In [4]:
sp = SamplingParams(temperature=0.2, max_tokens=128)
out = llm.generate(["Пояему трава зеленая"], sp)
print(out[0].outputs[0].text.strip())

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

? - Ответы на вопросы - Форумы - Беларусь - Беларусь.ру
Пояему трава зеленая?
Автор: Ксения, 23 апреля 2011 в Ответы на вопросы
Ксения 0
Почему трава зеленая? Или почему трава не красная, не желтая, не синяя?
Вот у меня есть такая мысль, но не уверен, что она верная. Может быть, кто-то подскажет


# 🎲 Параметры `SamplingParams` (с дефолтами)

## Основные

**`temperature`**  
Определяет "случайность" генерации.  
- Возможные значения: `0.0` и выше  
- По умолчанию: `1.0`  
> При `0.0` генерация становится детерминированной (жадной).

---

**`top_p`**  
Нуклеусная выборка: учитываются только токены, чья суммарная вероятность ≤ `p`.  
- Возможные значения: `0.0–1.0`  
- По умолчанию: `1.0` (отключено)  
> Часто используют `0.9`.

---

**`top_k`**  
Случайная выборка только из `k` наиболее вероятных токенов.  
- Возможные значения: целое число ≥1  
- По умолчанию: `-1` (отключено)

---

**`max_tokens`**  
Максимальное количество токенов для генерации (не включая вход).  
- Возможные значения: целое число ≥1  
- По умолчанию: `16`

---

**`presence_penalty`**  
Штраф за повторное появление токенов.  
- Возможные значения: любое число (обычно `0.0–2.0`)  
- По умолчанию: `0.0`

---

**`frequency_penalty`**  
Штраф за частое использование одних и тех же токенов.  
- Возможные значения: любое число (обычно `0.0–2.0`)  
- По умолчанию: `0.0`

---

**`repetition_penalty`**  
Коэффициент штрафа за повторения.  
- Возможные значения: число ≥0  
- По умолчанию: `1.0` (отключено)  
> Часто применяют `1.1–1.2`.

---

## Дополнительные

**`stop`**  
Список стоп-последовательностей (строки или токены). Генерация прерывается при встрече.  
- По умолчанию: `None`

---

**`ignore_eos`**  
Игнорировать ли токен конца последовательности `<eos>`.  
- Возможные значения: `True` / `False`  
- По умолчанию: `False`

---

**`best_of`**  
Генерация нескольких гипотез и выбор лучшей по logprob.  
- Возможные значения: целое число ≥1  
- По умолчанию: `1`  
⚠️ Увеличивает нагрузку на GPU.

---

**`logprobs`**  
Количество логарифмов вероятностей для возврата вместе с текстом.  
- Возможные значения: целое число ≥0  
- По умолчанию: `0` (не возвращает)

---

**`deterministic`**  
Фиксировать ли жадный поиск при temperature=0.  
- По умолчанию: `False`

---

**`seed`**  
Сид случайности именно для генерации (отдельно от `LLM(seed=...)`).  
- По умолчанию: `None`


In [6]:
prompts = ["Объясни простыми словами, что такое vLLM."]
outputs = llm.generate(
    prompts,
)

# Вывод результата
print(outputs[0].outputs[0].text.strip())

WARNING 08-19 12:10:46 [__init__.py:1625] Default sampling parameters have been overridden by the model's Hugging Face generation config recommended from the model creator. If this is not intended, please relaunch vLLM instance with `--generation-config vllm`.


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

vLLM - это система машинного обучения, которая используется для обработки


In [11]:
outputs[0].request_id, outputs[0].prompt,outputs[0].prompt_token_ids

('1',
 'Объясни простыми словами, что такое vLLM.',
 [73239,
  33594,
  127023,
  22621,
  129020,
  129129,
  91107,
  49707,
  11,
  47389,
  134322,
  348,
  4086,
  44,
  13])

In [12]:
outputs[0].outputs

[CompletionOutput(index=0, text=' vLLM - это система машинного обучения, которая используется для обработки', token_ids=(348, 4086, 44, 481, 67879, 134450, 130839, 38800, 143181, 11, 130407, 143248, 19849, 21229, 66086, 16748), cumulative_logprob=None, logprobs=None, finish_reason=length, stop_reason=None)]

# 📦 Структура результата генерации в vLLM

## 🔹 RequestOutput

У каждого объекта есть:
- **request_id** — уникальный ID запроса (строка)  
- **prompt** — исходный текст запроса  
- **prompt_token_ids** — список ID токенов входного промпта  
- **outputs** — список гипотез (обычно 1, если не указали `best_of > 1`)  

---

## 🔹 RequestOutput.outputs[i]

Каждая гипотеза содержит:
- **text** — сгенерированный текст (строка)  
- **token_ids** — список ID токенов ответа  
- **logprobs** — список логарифмов вероятностей (если в `SamplingParams(logprobs=N)`)  
- **cumulative_logprob** — суммарный logprob последовательности  
- **finish_reason** — причина остановки генерации:
  - "length" — достигнут `max_tokens`  
  - "stop" — встретили stop-sequence  
  - "eos_token" — модель выдала `<eos>`  
  - "abort" — досрочно завершено пользователем  

---

## 🔹 Если включить logprobs

Для каждого токена можно получить:
- **token_id** — ID токена  
- **logprob** — логарифм вероятности  
- **decoded_token** — символ или слово  
- **rank** — позиция токена по вероятности  
